# Projeto de Visualização de Dados e Business Intelligence [T3]
## Situação de Aprendizagem - Módulo 1 - Semana 13
**Aluno:** Amilcar  
**Turma:** [T3]  
**Base de Dados:** Human Resources (HR) - FreeSQL  

---
### 1. Contexto e Objetivos do Projeto
O objetivo deste estudo é realizar uma **Análise Exploratória de Dados (EDA)** sobre a estrutura salarial, cargos, departamentos e distribuição geográfica dos colaboradores da base HR do FreeSQL.

A análise busca responder às principais diretrizes de tomada de decisão do RH:
1. Identificar padrões e amplitudes de remuneração por cargo e setor;
2. Mapear a distribuição geográfica da força de trabalho e comparativos salariais continentais;
3. Avaliar a aderência entre **Média** e **Mediana** para compreender a assimetria salarial;
4. Fornecer gráficos descritivos (Boxplot, Histograma e Barras) com embasamento estatístico para a gestão de pessoas.

### 2. Importação das Bibliotecas e Configuração do Ambiente

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
IMG_DIR = BASE_DIR / 'img'
IMG_DIR.mkdir(parents=True, exist_ok=True)

### 3. Carga e Inspeção dos Dados Extraídos do FreeSQL
- `query_01.csv` / `Query_1.csv`: Salários por departamento e cargo (LEFT JOIN em DEPARTMENTS e JOBS).
- `query_02.csv` / `Query_2.csv`: Funcionários por região geográfica (LEFT JOIN em DEPARTMENTS, LOCATIONS, COUNTRIES e REGIONS).

In [ ]:
caminho_q1 = DATA_DIR / 'query_01.csv' if (DATA_DIR / 'query_01.csv').exists() else DATA_DIR / 'Query_1.csv'
caminho_q2 = DATA_DIR / 'query_02.csv' if (DATA_DIR / 'query_02.csv').exists() else DATA_DIR / 'Query_2.csv'

df_cargos = pd.read_csv(caminho_q1)
df_regioes = pd.read_csv(caminho_q2)

# Padronizar nomes de colunas
df_cargos.columns = [c.strip().upper() for c in df_cargos.columns]
df_regioes.columns = [c.strip().upper() for c in df_regioes.columns]

print(f'Total de registros na Query 1: {len(df_cargos)}')
print(f'Total de registros na Query 2: {len(df_regioes)}')
df_cargos.head()

### 4. Cálculo de Medidas Estatísticas Básicas
Cálculo da **Média**, **Mediana**, **Mínimo**, **Máximo**, **Desvio Padrão**, **Amplitude** e **Quartis (Q1 e Q3)** conforme requisitos da rubrica avaliativa.

In [ ]:
media = df_cargos['SALARIO'].mean()
mediana = df_cargos['SALARIO'].median()
minimo = df_cargos['SALARIO'].min()
maximo = df_cargos['SALARIO'].max()
desvio = df_cargos['SALARIO'].std()
amplitude = maximo - minimo
q25 = df_cargos['SALARIO'].quantile(0.25)
q75 = df_cargos['SALARIO'].quantile(0.75)

tabela_metricas = pd.DataFrame({
    'Métrica Estatística': ['Média Salarial', 'Mediana Salarial', 'Salário Mínimo', 'Salário Máximo', 'Desvio Padrão', 'Amplitude Salarial', '1º Quartil (25%)', '3º Quartil (75%)'],
    'Valor (USD)': [f'$ {media:,.2f}', f'$ {mediana:,.2f}', f'$ {minimo:,.2f}', f'$ {maximo:,.2f}', f'$ {desvio:,.2f}', f'$ {amplitude:,.2f}', f'$ {q25:,.2f}', f'$ {q75:,.2f}']
})
tabela_metricas

### 5. Análise Estatística por Departamento

In [ ]:
col_dept = 'NOME_DEPARTAMENTO' if 'NOME_DEPARTAMENTO' in df_cargos.columns else 'DEPARTAMENTO'
stats_dept = df_cargos.groupby(col_dept)['SALARIO'].agg(
    Qtd='count',
    Media='mean',
    Mediana='median',
    Minimo='min',
    Maximo='max',
    Desvio_Padrao='std'
).sort_values(by='Mediana', ascending=False).reset_index()

stats_dept

### 6. Análise Estatística por Região Geográfica

In [ ]:
stats_reg = df_regioes.groupby('REGIAO')['SALARIO'].agg(
    Qtd='count',
    Media='mean',
    Mediana='median',
    Minimo='min',
    Maximo='max',
    Desvio_Padrao='std'
).sort_values(by='Mediana', ascending=False).reset_index()

stats_reg

### 7. Gráfico 1: Boxplot dos Salários por Departamento
O **Boxplot** permite visualizar quartis, assimetria interna e valores atípicos (*outliers*), além da dispersão pontual de cada colaborador.

In [ ]:
plt.figure(figsize=(12, 6))
ordem = df_cargos.groupby(col_dept)['SALARIO'].median().sort_values(ascending=False).index

palette = sns.color_palette('Blues_r', n_colors=len(ordem))
ax = sns.boxplot(x=col_dept, y='SALARIO', hue=col_dept, data=df_cargos, order=ordem, palette=palette, legend=False, width=0.6)
sns.stripplot(x=col_dept, y='SALARIO', data=df_cargos, order=ordem, color='#2c3e50', alpha=0.5, jitter=0.2, size=6)

plt.title('Distribuição Salarial por Departamento (com Dispersão Individual)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Departamento', fontsize=11, fontweight='bold')
plt.ylabel('Salário Mensal (USD)', fontsize=11, fontweight='bold')
plt.xticks(rotation=35, ha='right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(IMG_DIR / 'boxplot_salarios_departamento.png', dpi=300)
plt.show()

### 8. Gráfico 2: Histograma com Curva KDE e Comparativo Média vs Mediana
Demonstra a assimetria positiva (*right-skewed*) da base: a **Média ($ 6.456,75)** é puxada para cima pelos altos executivos em relação à **Mediana ($ 6.150,00)**.

In [ ]:
plt.figure(figsize=(10, 5.5))
ax = sns.histplot(df_cargos['SALARIO'], bins=12, kde=True, color='#2980b9', edgecolor='#1c2833', alpha=0.65)

plt.axvline(media, color='#e74c3c', linestyle='--', linewidth=2.5, label=f'Média: $ {media:,.2f}')
plt.axvline(mediana, color='#27ae60', linestyle='-', linewidth=2.5, label=f'Mediana: $ {mediana:,.2f}')

plt.title('Distribuição Geral de Salários: Evidência de Assimetria Positiva', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Salário (USD)', fontsize=11, fontweight='bold')
plt.ylabel('Frequência de Colaboradores', fontsize=11, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.legend(fontsize=11, loc='upper right')
plt.tight_layout()
plt.savefig(IMG_DIR / 'histograma_distribuicao_salarial.png', dpi=300)
plt.show()

### 9. Gráfico 3: Comparativo Regional (Média Salarial vs Contagem de Pessoal)
Exibe a relação entre a remuneração média e o contingente alocado em **Americas** e **Europe**.

In [ ]:
resumo_reg = df_regioes.groupby('REGIAO')['SALARIO'].agg(['mean', 'count']).reset_index()

fig, ax1 = plt.subplots(figsize=(8.5, 5))
cor_barras = '#1f4e79'
bars = ax1.bar(resumo_reg['REGIAO'], resumo_reg['mean'], color=cor_barras, width=0.45, label='Média Salarial')
ax1.set_xlabel('Região Geográfica', fontsize=11, fontweight='bold')
ax1.set_ylabel('Média Salarial (USD)', color=cor_barras, fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=cor_barras)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for bar in bars:
    h = bar.get_height()
    ax1.annotate(f'$ {h:,.2f}', xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 5), textcoords='offset points', ha='center', va='bottom', fontweight='bold')

ax2 = ax1.twinx()
cor_linha = '#d35400'
ax2.plot(resumo_reg['REGIAO'], resumo_reg['count'], color=cor_linha, marker='o', linewidth=2.5, markersize=9, label='Total de Colaboradores')
ax2.set_ylabel('Total de Colaboradores', color=cor_linha, fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=cor_linha)
ax2.grid(False)

for i, row in resumo_reg.iterrows():
    ax2.annotate(f"{row['count']} func.", xy=(row['REGIAO'], row['count']), xytext=(0, 8), textcoords='offset points', ha='center', va='bottom', color=cor_linha, fontweight='bold')

plt.title('Comparativo Regional: Média Salarial vs Contagem de Pessoal', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(IMG_DIR / 'media_salarial_regiao.png', dpi=300)
plt.show()

### 10. Conclusões e Principais Insights Estratégicos para o RH
1. **Estratificação Setorial:** O departamento *Executive* tem a maior média ($ 19.333,33) e máxima ($ 24.000,00), enquanto *Shipping* e *Purchasing* concentram a maior parte dos cargos de entrada com salários a partir de $ 2.100,00.
2. **Distorção Regional:** A Europa apresenta média mais alta ($ 8.916,67) que as Américas ($ 5.191,66) não por disparidade salarial arbitrária, mas sim pelo perfil funcional: a Europa abriga apenas postos de Vendas plenos/sêniores em Oxford e RP em Munique, sem a presença do centro de distribuição (Shipping), que fica alocado na Califórnia.
3. **Uso da Mediana em Decisões de RH:** Devido à assimetria positiva, a Mediana ($ 6.150,00) deve ser adotada como métrica central em planos de cargos e salários para evitar distorções induzidas pelos salários da diretoria.
4. **Limitação da Análise:** Os dados representam apenas a remuneração base, não considerando comissões (`COMMISSION_PCT`), benefícios e diferenças de custo de vida entre países.